# topk-predictions — ex1: top-5 accuracy on a batch of logits

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `topk-predictions`. Running the final beacon cell reports progress against the `Eval: topk predictions` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Eval: topk predictions` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`topk-predictions`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "topk-predictions"
DD_SUBTOPIC = "Eval: topk predictions"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## logits.topk for top-k accuracy — quick refresher

`logits.topk(k, dim=-1)` returns a named tuple `(values, indices)` where `values` are the `k` largest entries along `dim` and `indices` are their positions. For classification logits `(B, num_classes)`, `topk(5, dim=-1).indices` gives the `(B, 5)` tensor of the model's top-5 predicted class ids per sample.

**Top-k accuracy.** A prediction is 'top-5 correct' if the true label appears anywhere in the top-5 predicted ids: `(topk_indices == label.unsqueeze(-1)).any(dim=-1)`. Top-1 accuracy is the special case `k=1`.

ImageNet and many other benchmarks report top-1 AND top-5 because the long tail of classes is genuinely ambiguous — getting it in the top 5 is a useful signal even when the argmax is wrong.

### Exercise 1 — top-5 accuracy on a batch of logits

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `logits.topk(k=5, dim=-1).indices` then `any` over the k axis to compute top-5 classification accuracy from a batch of logits and ground-truth labels.
> Keywords: topk, accuracy, eval, classification
> ```

**KCs targeted:** `topk-returns-values-and-indices`, `topk-membership-test`

Implement `ex1_top5_accuracy(logits, labels)`. The standard ImageNet-style top-5 evaluation:

1. `logits` has shape `(B, num_classes)`.
2. `labels` has shape `(B,)` and dtype `long`.
3. Use `logits.topk(k=5, dim=-1)` — call the result `topk_out`. Use `topk_out.indices` (shape `(B, 5)`) for the predicted class ids.
4. A sample is correct if its true label appears in its top-5 indices. Use broadcasting: `(topk_out.indices == labels.unsqueeze(-1)).any(dim=-1)`.
5. Return the mean of that boolean tensor (as a float scalar tensor) — the top-5 accuracy in `[0, 1]`.

Input: `logits: (B, C)`, `labels: (B,) long`.
Output: scalar tensor, top-5 accuracy.

In [ ]:
def ex1_top5_accuracy(logits: Tensor, labels: Tensor) -> Tensor:
    topk_out = logits.topk(k=5, dim=-1)
    correct = (topk_out.indices == labels.unsqueeze(-1)).any(dim=-1)
    return correct.float().mean()


<details><summary>Solution</summary>

```python
def ex1_top5_accuracy(logits: Tensor, labels: Tensor) -> Tensor:
    topk_out = logits.topk(k=5, dim=-1)
    correct = (topk_out.indices == labels.unsqueeze(-1)).any(dim=-1)
    return correct.float().mean()
```

**The named-tuple return.** `logits.topk(5, dim=-1)` returns `torch.return_types.topk(values=..., indices=...)`. Always access with `.values` / `.indices`, never positional indexing — the named form is self-documenting.

**`labels.unsqueeze(-1)` is the broadcast trick.** Labels are `(B,)` and top-k indices are `(B, 5)`. Adding the trailing size-1 axis to labels (`(B, 1)`) lets them broadcast across the k=5 dimension for the equality test.

**Why `.float().mean()`.** The `correct` tensor is `bool`. `mean` on a bool tensor is undefined in many PyTorch versions; cast to float first so the mean is the fraction of correct predictions.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()